# Notebook 01 — Python Data Validation

This notebook loads the analysis-ready TCGA-BRCA cohort produced by the R cleaning pipeline (`R/02_clean_tcga_cdr.R`) and performs a basic data validation in Python. The goals are to:

1. Confirm the dataset dimensions and variable types.
2. Quantify missingness across all columns.
3. Inspect the distributions of key variables before any modelling is carried out.

**Input:** `data/processed/brca_survival_clean.csv`  
**Produced by:** `R/02_clean_tcga_cdr.R`  
**Next step:** `notebooks/02_ml_risk_prediction_demo.ipynb`

In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data/processed")

df = pd.read_csv(DATA_DIR / "brca_survival_clean.csv")

print("Shape:", df.shape)
display(df.head())
display(df.dtypes)

Shape: (1083, 13)


,patient_id,cancer_type,age,gender,race,menopause_status,histological_type,stage_raw,os_event,os_time_days,os_time_months,stage_group,age_group
0,TCGA-3C-AAAU,BRCA,55,FEMALE,WHITE,Pre (<6 months since LMP AND no prior bilatera...,Infiltrating Lobular Carcinoma,Stage X,0,4047,132.950066,NaN,50-64
1,TCGA-3C-AALI,BRCA,50,FEMALE,BLACK OR AFRICAN AMERICAN,Post (prior bilateral ovariectomy OR >12 mo si...,Infiltrating Ductal Carcinoma,Stage IIB,0,4005,131.570302,Stage II,50-64
2,TCGA-3C-AALJ,BRCA,62,FEMALE,BLACK OR AFRICAN AMERICAN,Post (prior bilateral ovariectomy OR >12 mo si...,Infiltrating Ductal Carcinoma,Stage IIB,0,1474,48.423127,Stage II,50-64
3,TCGA-3C-AALK,BRCA,52,FEMALE,BLACK OR AFRICAN AMERICAN,NaN,Infiltrating Ductal Carcinoma,Stage IA,0,1448,47.568988,Stage I,50-64
4,TCGA-4H-AAAK,BRCA,50,FEMALE,WHITE,Post (prior bilateral ovariectomy OR >12 mo si...,Infiltrating Lobular Carcinoma,Stage IIIA,0,348,11.432326,Stage III,50-64


patient_id               str
cancer_type              str
age                    int64
gender                   str
race                     str
menopause_status         str
histological_type        str
stage_raw                str
os_event               int64
os_time_days           int64
os_time_months       float64
stage_group              str
age_group                str
dtype: object

## Missingness

Missing data is quantified for every column. The columns with the highest missing rates are:

- **`race`** and **`menopause_status`**: both 7.7 % — these are reported inconsistently in the TCGA clinical data and will be imputed inside the ML pipeline.
- **`stage_group`**: 2.2 % (n = 24) — these patients have `stage_raw = "Stage X"`, a TCGA coding artefact that does not map to any standard AJCC stage group. They are excluded from the Cox regression model and from the ML modelling in notebook 02.
- **`stage_raw`**: 0.5 % (n = 5) — a small number of patients with no stage information at all.

Age, survival outcome, and follow-up time are complete for all 1 083 patients.

In [2]:
missingness = (
    df.isna()
    .sum()
    .reset_index(name="missing_n")
    .rename(columns={"index": "variable"})
)

missingness["missing_percent"] = round(
    100 * missingness["missing_n"] / len(df), 1
)

missingness = missingness.sort_values("missing_percent", ascending=False)

display(missingness)

# missingness.to_csv(DATA_DIR / "python_missingness_summary.csv", index=False)

,variable,missing_n,missing_percent
4,race,83,7.7
5,menopause_status,83,7.7
11,stage_group,24,2.2
7,stage_raw,5,0.5
6,histological_type,1,0.1
3,gender,0,0.0
2,age,0,0.0
1,cancer_type,0,0.0
0,patient_id,0,0.0
8,os_event,0,0.0


## Distribution checks

Univariate distributions are inspected to detect data quality issues before modelling. Key observations:

- **Overall survival events:** 151 of 1 083 patients (14 %) experienced death — a low event rate typical of TCGA-BRCA, which benefits from favourable prognosis and variable follow-up length.
- **Follow-up time:** Median ≈ 28.3 months; maximum ≈ 282.7 months (~23.6 years), consistent with long-term TCGA follow-up in earlier cohorts.
- **Stage distribution:** Stage II is the largest group (n = 612); Stage IV is small (n = 20), which limits the precision of Stage IV estimates throughout the analysis.
- **Race / menopause status:** Small `"[Not Evaluated]"` categories (3 and 6 patients respectively) are retained as nominal levels and handled by one-hot encoding in notebook 02.

In [3]:
display(df["os_event"].value_counts(dropna=False))
display(df["os_time_months"].describe())
display(df["stage_group"].value_counts(dropna=False))
display(df["race"].value_counts(dropna=False))
display(df["menopause_status"].value_counts(dropna=False))
display(df["histological_type"].value_counts(dropna=False))

os_event
0    932
1    151
Name: count, dtype: int64

count    1083.000000
mean       41.412621
std        39.171039
min         0.032852
25%        15.210250
50%        28.252300
75%        55.453351
max       282.687254
Name: os_time_months, dtype: float64

stage_group
Stage II     612
Stage III    245
Stage I      182
NaN           24
Stage IV      20
Name: count, dtype: int64

race
WHITE                               756
BLACK OR AFRICAN AMERICAN           181
NaN                                  83
ASIAN                                59
[Not Evaluated]                       3
AMERICAN INDIAN OR ALASKA NATIVE      1
Name: count, dtype: int64

menopause_status
Post (prior bilateral ovariectomy OR >12 mo since LMP with no prior hysterectomy)               693
Pre (<6 months since LMP AND no prior bilateral ovariectomy AND not on estrogen replacement)    228
NaN                                                                                              83
Peri (6-12 months since last menstrual period)                                                   39
Indeterminate (neither Pre or Postmenopausal)                                                    34
[Not Evaluated]                                                                                   6
Name: count, dtype: int64

histological_type
Infiltrating Ductal Carcinoma       774
Infiltrating Lobular Carcinoma      202
Other, specify                       45
Mixed Histology (please specify)     30
Mucinous Carcinoma                   16
Metaplastic Carcinoma                 9
Medullary Carcinoma                   5
Infiltrating Carcinoma NOS            1
NaN                                   1
Name: count, dtype: int64